# Day 014：从 Full SFT 入口进入训练链路

本 Notebook 与 `day-014-2026-08-31.md` 配套，复现今天已经学习的范围：SFT 样本如何从多轮 conversation 变成 `(input_ids, labels)`，以及 `train_full_sft.py` 在进入训练循环前如何准备模型、数据和优化器。

恢复点：`trainer/train_full_sft.py` 第 158 行 `for epoch ...`。DataLoader、batch 训练、forward、backward 和 `optimizer.step()` 留到下一天。

## 1. 一条 SFT 样本的层级

JSONL 一行是一个 item；item 的 `conversations` 是一组有顺序的 message。多条 user、assistant、tool 消息会先由 chat template 拼成一个 prompt 字符串，再整体分词成一组 `input_ids`。它们不是四个独立训练样本。

In [ ]:
import json

raw_tools = '[{"type":"function","function":{"name":"get_weather","description":"查询天气"}}]'
raw_tool_calls = '[{"name":"get_weather","arguments":{"city":"北京"}}]'

tools = json.loads(raw_tools)
tool_calls = json.loads(raw_tool_calls)
messages = [
    {"role": "system", "content": "你可以调用工具。", "tools": raw_tools},
    {"role": "user", "content": "北京天气怎么样？"},
    {"role": "assistant", "content": "", "tool_calls": tool_calls},
    {"role": "tool", "content": "{\"weather\": \"晴\"}"}
]

print(type(messages).__name__)
print(type(messages[0]).__name__)
print(type(tools).__name__)
print(type(messages[2]['tool_calls']).__name__)
print(type(messages[0]['tools']).__name__)  # system message 中仍保留原始字符串
assert isinstance(messages, list)
assert isinstance(messages[0], dict)
assert isinstance(tools, list)
assert isinstance(messages[2]['tool_calls'], list)
assert isinstance(messages[0]['tools'], str)

## 2. 使用本地 tokenizer 生成工具对话 prompt

下面的路径假设 Notebook 从 `learning-journal/notebooks` 打开；如果工作目录不同，请调整路径。`tokenize=False` 只生成字符串，`add_generation_prompt=False` 适合已经包含 assistant 目标的 SFT 样本。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('../../minimind/model')
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
    tools=tools
)
print(prompt)
assert isinstance(prompt, str)
assert '<tool_call>' in prompt
assert '<tool_response>' in prompt


## 3. 第 111～112 行：分词、截断和 padding

`self.tokenizer(prompt).input_ids` 在没有 `return_tensors='pt'` 时是一维 `list[int]`。先保留前 `max_length` 个，再补 `pad_token_id`；这一步还没有计算 loss。

In [ ]:
max_length = 16
toy_prompt_ids = list(range(7))
input_ids = toy_prompt_ids[:max_length]
pad_token_id = 0
input_ids += [pad_token_id] * (max_length - len(input_ids))
print(input_ids)
print('长度：', len(input_ids))
assert len(input_ids) == max_length
assert input_ids[-1] == pad_token_id

# 过长时先截断，之后不再补 pad
too_long = list(range(20))[:max_length]
too_long += [pad_token_id] * (max_length - len(too_long))
assert len(too_long) == max_length
assert too_long == list(range(16))

## 4. `generate_labels()`：只让 assistant 区域计入 loss

`labels` 与 `input_ids` 等长。初始化全为 `-100`，它是交叉熵的忽略标记，不是词表 token。找到 `<|im_start|>assistant\n` 后，把 assistant 正文以及 `<|im_end|>\n` 的 token ID 写入 labels；system、user、tool、padding 仍为 `-100`。

In [ ]:
bos_id = [1, 8, 9]
eos_id = [2, 0]
input_ids = [4, 5, 1, 8, 9, 6, 7, 2, 0, 4]
max_length = len(input_ids)
labels = [-100] * len(input_ids)
i = 0
while i < len(input_ids):
    if input_ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(input_ids):
            if input_ids[end:end + len(eos_id)] == eos_id:
                break
            end += 1
        for j in range(start, min(end + len(eos_id), max_length)):
            labels[j] = input_ids[j]
        i = end + len(eos_id) if end < len(input_ids) else len(input_ids)
    else:
        i += 1

print('input_ids:', input_ids)
print('labels:   ', labels)
assert labels == [-100, -100, -100, -100, -100, 6, 7, 2, 0, -100]

## 5. `__getitem__()` 的返回值

最后将两个 Python list 转为 `torch.long`。`input_ids` 需要整数类型作为 Embedding 查表索引；`labels` 需要整数类别 ID，`-100` 继续表示忽略。这里仍然是单条样本，没有 batch 维度，也没有 loss。

In [ ]:
import torch

sample_return = (
    torch.tensor(input_ids, dtype=torch.long),
    torch.tensor(labels, dtype=torch.long)
)
print(type(sample_return).__name__)
print(sample_return[0].dtype, sample_return[0].shape)
print(sample_return[1].dtype, sample_return[1].shape)
assert isinstance(sample_return, tuple)
assert sample_return[0].dtype == torch.long
assert tuple(sample_return[0].shape) == (10,)

## 6. 训练准备阶段的关键分支

普通单进程默认值：`from_resume=0`、`dtype='bfloat16'`、`use_compile=0`、`dist.is_initialized()=False`。因此不恢复 checkpoint、不启用 GradScaler、不编译、不做 DDP 包装。优化器仍然会创建，但不会在创建时更新参数。

In [ ]:
from_resume = 0
dtype = 'bfloat16'
use_compile = 0
distributed_initialized = False

ckp_data = None if from_resume == 0 else {'epoch': 1, 'step': 320}
start_epoch, start_step = 0, 0
if ckp_data:
    start_epoch = ckp_data['epoch']
    start_step = ckp_data.get('step', 0)

scaler_enabled = dtype == 'float16'
compile_enabled = use_compile == 1
ddp_enabled = distributed_initialized
print({'start_epoch': start_epoch, 'start_step': start_step, 'scaler_enabled': scaler_enabled, 'compile_enabled': compile_enabled, 'ddp_enabled': ddp_enabled})
assert (start_epoch, start_step) == (0, 0)
assert scaler_enabled is False
assert compile_enabled is False
assert ddp_enabled is False

## 7. SGD、Adam、AdamW 的概念对照

SGD 主要按当前梯度更新：`parameter -= lr * gradient`。Adam 用 `m` 记住历史方向、用 `v` 记住梯度尺度，并通过偏差修正得到 `m_hat`、`v_hat`。AdamW 再额外做独立的权重衰减，让参数绝对值轻微向 0 收缩。MiniMind 使用 `optim.AdamW(model.parameters(), lr=1e-5)`。

In [ ]:
parameter = 10.0
gradient = 2.0
learning_rate = 0.1
sgd_parameter = parameter - learning_rate * gradient
print('SGD 更新后：', sgd_parameter)
assert sgd_parameter == 9.8

m_old, current_gradient, beta1 = 3.0, -1.0, 0.5
m_new = beta1 * m_old + (1 - beta1) * current_gradient
v_old, beta2 = 8.0, 0.5
v_new = beta2 * v_old + (1 - beta2) * current_gradient ** 2
print('Adam m：', m_new, 'v：', v_new)
assert m_new == 1.0
assert v_new == 4.5

## 今日恢复点

已完成模型、数据集、labels、优化器和可选恢复/编译/分布式包装的入口准备。下一天从 `trainer/train_full_sft.py` 第 158 行开始，阅读 `for epoch in range(...)`、随机索引、`SkipBatchSampler`、`DataLoader`，然后进入 `train_epoch()` 的真实训练步骤。